# Visualization Gallery

The [previous EDA notebook](exploratory-data-analysis.ipynb) profiled the messy
`customers.csv` numerically. This companion notebook is a **chart-type catalogue**:
one real plot per visualization technique, against that same dataset, so you have
a reference for *which chart answers which question*.

Several of these chart types have **no primitive in `plotters` itself** — box
plots, violins, heatmaps, pair plots, ECDF and Q–Q plots. For those we use
[`plotters-statistical`](https://crates.io/crates/plotters-statistical), which
adds them as native `plotters` series and figures (so they compose with
`ChartBuilder` / `draw_series` exactly like the built-ins). Histograms, bar
charts, and scatter/line plots use `plotters`' own built-ins directly.

The closing [chart-selection table](#which-chart-for-which-question) summarises
when to reach for each one.

In [ ]:
:dep polars = { version = "0.44", features = ["lazy", "ndarray", "parquet", "strings"] }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
:dep plotters-statistical = { version = "0.2.0" }
use polars::prelude::*;
use plotters::prelude::*;

let df = CsvReadOptions::default()
    .with_has_header(true)
    .try_into_reader_with_file_path(Some("/book/data/customers.csv".into()))?
    .finish()?;
println!("shape = {:?}", df.shape());
let numeric = ["age", "income", "tenure_months", "monthly_charge"];

In [ ]:
// Helper: pull a numeric column (non-null) into a Vec<f64> via the ndarray bridge.
fn col_f64(df: &DataFrame, name: &str) -> PolarsResult<Vec<f64>> {
    let m = df.clone().lazy()
        .filter(col(name).is_not_null())
        .select([col(name).cast(DataType::Float64)])
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    Ok((0..m.nrows()).map(|i| m[[i, 0]]).collect())
}

// Helper: linear-interpolated quantile of a pre-sorted slice.
fn quantile(sorted: &[f64], p: f64) -> f64 {
    let n = sorted.len();
    if n == 0 { return f64::NAN; }
    let idx = (n as f64 - 1.0) * p;
    let lo = idx.floor() as usize;
    let hi = idx.ceil() as usize;
    let frac = idx - lo as f64;
    sorted[lo] * (1.0 - frac) + sorted[hi] * frac
}
println!("helpers defined");

## 1. Histogram — distribution shape

The first question about any numeric column: what's its *shape*? Histograms bin
the values and count them. A 2×2 grid, one per numeric column (income clipped
below 200k so the ~900k outlier doesn't flatten the rest):

In [ ]:
evcxr_figure((760, 540), |root| {
    root.fill(&WHITE)?;
    for (area, name) in root.split_evenly((2, 2)).iter().zip(numeric.iter()) {
        let raw = col_f64(&df, name)?;
        let vals: Vec<f64> = if *name == "income" {
            raw.into_iter().filter(|v| *v < 200_000.0).collect()
        } else { raw };
        let lo = vals.iter().cloned().fold(f64::INFINITY, f64::min);
        let hi = vals.iter().cloned().fold(f64::NEG_INFINITY, f64::max);
        let nbins = 8usize;
        let width = ((hi - lo) / nbins as f64).max(1e-9);
        let mut counts = vec![0u32; nbins];
        for v in &vals {
            let mut b = ((v - lo) / width) as usize;
            if b >= nbins { b = nbins - 1; }
            counts[b] += 1;
        }
        let maxc = *counts.iter().max().unwrap_or(&1);
        let mut chart = ChartBuilder::on(area)
            .caption(*name, ("sans-serif", 15))
            .margin(6).x_label_area_size(26).y_label_area_size(30)
            .build_cartesian_2d(lo..hi, 0u32..(maxc + 1))?;
        chart.configure_mesh().disable_mesh().draw()?;
        chart.draw_series(counts.iter().enumerate().map(|(i, &c)| {
            let x0 = lo + i as f64 * width;
            Rectangle::new([(x0, 0), (x0 + width * 0.9, c)], BLUE.filled())
        }))?;
    }
    Ok(())
})

## 2. Box & violin plots — quartiles, outliers & density

A box plot summarises a distribution by its five-number summary (whiskers, Q1,
median, Q3) and marks IQR outliers; a **violin** adds a mirrored density curve so
you also see the *shape* (skew, multiple modes) a box hides. `plotters` has no
primitive for either, so we use
[`plotters-statistical`](https://crates.io/crates/plotters-statistical)'s
`BoxPlotSeries` / `ViolinPlotSeries` — each takes `Vec<(position, samples)>` and
draws like any other series. One panel per numeric column (each keeps its own
y-scale, since the ranges differ by orders of magnitude):

In [ ]:
use plotters_statistical::BoxPlotSeries;

evcxr_figure((760, 540), |root| {
    root.fill(&WHITE)?;
    for (area, name) in root.split_evenly((2, 2)).iter().zip(numeric.iter()) {
        let v = col_f64(&df, name)?;
        let (lo, hi) = (v.iter().cloned().fold(f64::INFINITY, f64::min),
                        v.iter().cloned().fold(f64::NEG_INFINITY, f64::max));
        let pad = (hi - lo) * 0.1 + 1e-9;
        let mut chart = ChartBuilder::on(area)
            .caption(*name, ("sans-serif", 15))
            .margin(6).x_label_area_size(8).y_label_area_size(46)
            .build_cartesian_2d(0.5f64..1.5f64, (lo - pad)..(hi + pad))?;
        chart.configure_mesh().disable_x_mesh().draw()?;
        // A single group at x = 1.0; the series computes quartiles + outliers itself.
        chart.draw_series(BoxPlotSeries::from_samples(vec![(1.0f64, v)])?.width(60))?;
    }
    Ok(())
})

The same four columns as **violins**, with the box overlaid (`.show_box(true)`).
The bulge shows where values concentrate — note how `income`'s violin is pinched
near the bottom with a long thin neck toward the 900k outlier, the skew a box
alone only hints at:

In [ ]:
use plotters_statistical::ViolinPlotSeries;

evcxr_figure((760, 540), |root| {
    root.fill(&WHITE)?;
    for (area, name) in root.split_evenly((2, 2)).iter().zip(numeric.iter()) {
        let v = col_f64(&df, name)?;
        let (lo, hi) = (v.iter().cloned().fold(f64::INFINITY, f64::min),
                        v.iter().cloned().fold(f64::NEG_INFINITY, f64::max));
        let pad = (hi - lo) * 0.1 + 1e-9;
        let mut chart = ChartBuilder::on(area)
            .caption(*name, ("sans-serif", 15))
            .margin(6).x_label_area_size(8).y_label_area_size(46)
            .build_cartesian_2d(0.5f64..1.5f64, (lo - pad)..(hi + pad))?;
        chart.configure_mesh().disable_x_mesh().draw()?;
        chart.draw_series(ViolinPlotSeries::from_samples(vec![(1.0f64, v)])?.width(70).show_box(true))?;
    }
    Ok(())
})

## 3. Correlation heatmap

How strongly does each numeric feature move with the others?
`plotters-statistical`'s `CorrelationHeatmap::from_columns` takes the columns and
their labels, computes the Pearson matrix itself, and renders it with a diverging
scale and the value printed in each cell. This is the companion to the pair plot
below — the heatmap says *how strongly* related, the pair plot shows *what the
relationship looks like*. (We drop rows with any missing value so every pair is
computed on the same aligned observations.)

In [ ]:
use plotters_statistical::stats::CorrelationMethod;
use plotters_statistical::CorrelationHeatmap;

// One Vec<f64> per numeric column, rows with any null dropped so all columns
// share the same aligned observations. (Building the data outside the closure
// keeps a nameable type; the raw ndarray can't be named across cells.)
let corr_cols: Vec<Vec<f64>> = {
    let a = df.clone().lazy()
        .select(numeric.iter().map(|c| col(*c).cast(DataType::Float64)).collect::<Vec<_>>())
        .drop_nulls(None)
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    (0..a.ncols()).map(|j| (0..a.nrows()).map(|i| a[[i, j]]).collect()).collect()
};

evcxr_figure((560, 500), |root| {
    root.fill(&WHITE)?;
    let labels: Vec<String> = numeric.iter().map(|s| s.to_string()).collect();
    CorrelationHeatmap::from_columns(&corr_cols, labels, CorrelationMethod::Pearson)?
        .title("Pearson correlation")
        .precision(2)
        .draw(&root)?;
    Ok(())
})

## 4. Pair plot (scatter-plot matrix)

Every numeric column against every other: the **diagonal** shows each column's
histogram, the **off-diagonal** panels are scatter plots coloured by `churned`,
so you see the *shape* of each pairwise relationship the heatmap above only
scored. `plotters-statistical`'s `PairPlot::new(columns, labels)` builds the whole
grid; `.hue(group)` colours points by class and `.diagonal(Diagonal::Histogram)`
picks the diagonal panel:

In [ ]:
use plotters_statistical::figures::Diagonal;
use plotters_statistical::PairPlot;

// Per-column values + the churned class per row, null rows dropped jointly so the
// columns and the hue vector stay aligned.
let (pp_cols, pp_hue): (Vec<Vec<f64>>, Vec<usize>) = {
    let a = df.clone().lazy()
        .select(numeric.iter().map(|c| col(*c).cast(DataType::Float64))
            .chain(std::iter::once(col("churned").cast(DataType::Float64)))
            .collect::<Vec<_>>())
        .drop_nulls(None)
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    let ncols = numeric.len();
    let cols = (0..ncols).map(|j| (0..a.nrows()).map(|i| a[[i, j]]).collect()).collect();
    let hue = (0..a.nrows()).map(|i| a[[i, ncols]] as usize).collect();
    (cols, hue)
};

evcxr_figure((820, 760), |root| {
    root.fill(&WHITE)?;
    let labels: Vec<String> = numeric.iter().map(|s| s.to_string()).collect();
    PairPlot::new(pp_cols.clone(), labels)
        .diagonal(Diagonal::Histogram)
        .hue(pp_hue.clone())
        .marker_radius(2)
        .draw(&root)?;
    Ok(())
})

## 5. Bar chart — categorical value counts

For a categorical column, a bar chart of value counts. We normalise `city` first
(trim + lower-case, as the [ETL chapter](../01c-etl/data-preparation.ipynb) does)
so the messy variants collapse, then count:

In [ ]:
let counts = df.clone().lazy()
    .with_columns([col("city").str().strip_chars(lit(" ")).str().to_lowercase().alias("city")])
    .group_by([col("city")])
    .agg([len().alias("n")])
    .sort(["n"], SortMultipleOptions::default().with_order_descending(true))
    .collect()?;
let cats: Vec<String> = counts.column("city")?.str()?.into_iter().map(|o| o.unwrap_or("").to_string()).collect();
let ns: Vec<f64> = col_f64(&counts, "n")?;

evcxr_figure((560, 360), |root| {
    root.fill(&WHITE)?;
    let n = cats.len();
    let maxc = ns.iter().cloned().fold(0.0, f64::max);
    let mut chart = ChartBuilder::on(&root)
        .caption("city — value counts", ("sans-serif", 16))
        .margin(10).x_label_area_size(30).y_label_area_size(35)
        .build_cartesian_2d(0f64..n as f64, 0f64..(maxc + 1.0))?;
    chart.configure_mesh().disable_x_mesh()
        .x_labels(n).x_label_formatter(&|x| cats.get(*x as usize).cloned().unwrap_or_default())
        .draw()?;
    chart.draw_series(ns.iter().enumerate().map(|(i, &c)| {
        Rectangle::new([(i as f64 + 0.1, 0.0), (i as f64 + 0.9, c)], GREEN.mix(0.7).filled())
    }))?;
    Ok(())
})

## 6. Bar chart — class balance

The same bar-chart family, applied to the target. This is the imbalance the ETL
and Evaluation chapters keep in mind — here as a chart rather than a printed
count:

In [ ]:
let bal = df.clone().lazy()
    .group_by([col("churned")]).agg([len().alias("n")])
    .sort(["churned"], SortMultipleOptions::default())
    .collect()?;
let bns: Vec<f64> = col_f64(&bal, "n")?;

evcxr_figure((420, 340), |root| {
    root.fill(&WHITE)?;
    let maxc = bns.iter().cloned().fold(0.0, f64::max);
    let mut chart = ChartBuilder::on(&root)
        .caption("class balance (churned)", ("sans-serif", 16))
        .margin(10).x_label_area_size(30).y_label_area_size(35)
        .build_cartesian_2d(0f64..2f64, 0f64..(maxc + 2.0))?;
    chart.configure_mesh().disable_x_mesh()
        .x_labels(2).x_label_formatter(&|x| if *x < 0.5 { "0 (kept)".into() } else if *x < 1.5 { "1 (churned)".into() } else { "".into() })
        .draw()?;
    chart.draw_series(bns.iter().enumerate().map(|(i, &c)| {
        let color = if i == 0 { BLUE.mix(0.7) } else { RGBColor(230, 140, 0).mix(0.9) };
        Rectangle::new([(i as f64 + 0.1, 0.0), (i as f64 + 0.9, c)], color.filled())
    }))?;
    Ok(())
})

## 7. Scatter plot with outliers flagged

The [first EDA notebook](exploratory-data-analysis.ipynb) flagged income outliers
by the IQR rule but only *printed* them. Here we render `income` vs
`monthly_charge` as a scatter, colouring IQR-flagged income points red — the
~900k customer stands out immediately on the right:

In [ ]:
let sm: Vec<Vec<f64>> = {
    let a = df.clone().lazy()
        .filter(col("income").is_not_null())
        .select([col("income").cast(DataType::Float64), col("monthly_charge").cast(DataType::Float64)])
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    (0..a.nrows()).map(|i| (0..a.ncols()).map(|j| a[[i, j]]).collect()).collect()
};
let rows = sm.len();
let fence = {
    let mut inc: Vec<f64> = (0..rows).map(|i| sm[i][0]).collect();
    inc.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let (q1, q3) = (quantile(&inc, 0.25), quantile(&inc, 0.75));
    q3 + 1.5 * (q3 - q1)
};

evcxr_figure((620, 420), |root| {
    root.fill(&WHITE)?;
    let xhi = (0..rows).map(|i| sm[i][0]).fold(0.0, f64::max) * 1.05;
    let yhi = (0..rows).map(|i| sm[i][1]).fold(0.0, f64::max) * 1.10;
    let mut chart = ChartBuilder::on(&root)
        .caption("income vs monthly_charge (IQR outliers in red)", ("sans-serif", 15))
        .margin(10).x_label_area_size(36).y_label_area_size(52)
        .build_cartesian_2d(0f64..xhi, 0f64..yhi)?;
    chart.configure_mesh().x_desc("income").y_desc("monthly_charge").draw()?;
    chart.draw_series((0..rows).map(|i| {
        let flagged = sm[i][0] > fence;
        let style = if flagged { RED.filled() } else { BLUE.mix(0.6).filled() };
        Circle::new((sm[i][0], sm[i][1]), if flagged { 5 } else { 3 }, style)
    }))?;
    Ok(())
})

## 8. Missingness heatmap

Per-column null *counts* (from the first notebook) don't show whether missing
values cluster in particular rows. `plotters-statistical`'s `MissingnessHeatmap`
takes each column as a `Vec<Option<f64>>` (`None` = missing) and draws a
rows × columns grid — one cell per value, dark where it's missing — making the
pattern visible at a glance:

In [ ]:
use plotters_statistical::MissingnessHeatmap;

let allcols = ["customer_id", "age", "income", "city", "tenure_months", "monthly_charge", "churned"];
// One Vec<Option<f64>> per column: None where the value is null, Some(_) otherwise.
// (The value itself is irrelevant to missingness, so we use the null mask as f64
// and map it back to Option, reusing the proven to_ndarray bridge.)
let miss_cols: Vec<Vec<Option<f64>>> = {
    let a = df.clone().lazy()
        .select(allcols.iter().map(|c| col(*c).is_null().cast(DataType::Float64).alias(*c)).collect::<Vec<_>>())
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    (0..a.ncols())
        .map(|j| (0..a.nrows()).map(|i| if a[[i, j]] > 0.5 { None } else { Some(0.0) }).collect())
        .collect()
};
let miss_labels: Vec<String> = allcols.iter().map(|s| s.to_string()).collect();

evcxr_figure((620, 620), |root| {
    root.fill(&WHITE)?;
    MissingnessHeatmap::from_columns(&miss_cols, miss_labels.clone())?
        .title("missingness (dark = missing)")
        .draw(&root)?;
    Ok(())
})

## 9. ECDF & Q–Q plot

An **ECDF** plots each sorted value against the fraction of data at or below it —
an often-skipped complement to the histogram that makes distribution shape precise
with no binning choice to distort it. `plotters-statistical`'s `Ecdf::from_data`
draws it as a series (it can also add a DKW confidence band). One per numeric
column:

In [ ]:
use plotters_statistical::style::palette_color;
use plotters_statistical::Ecdf;

evcxr_figure((760, 540), |root| {
    root.fill(&WHITE)?;
    for (area, name) in root.split_evenly((2, 2)).iter().zip(numeric.iter()) {
        let v = col_f64(&df, name)?;
        let (lo, hi) = (v.iter().cloned().fold(f64::INFINITY, f64::min),
                        v.iter().cloned().fold(f64::NEG_INFINITY, f64::max));
        let mut chart = ChartBuilder::on(area)
            .caption(*name, ("sans-serif", 15))
            .margin(6).x_label_area_size(26).y_label_area_size(34)
            .build_cartesian_2d(lo..hi, 0f64..1.05f64)?;
        chart.configure_mesh().y_desc("F(x)").draw()?;
        chart.draw_series(std::iter::once(Ecdf::from_data(&v)?.color(palette_color(0))))?;
    }
    Ok(())
})

A **normal Q–Q plot** goes one step further: it plots each column's sorted values
against the quantiles a normal distribution would produce. Points on the diagonal
reference line mean "looks normal"; systematic bends mean skew or heavy tails.
`QqPlot::from_data` draws the points and the reference line — watch `income` bend
hard away from the line (its 900k outlier makes it strongly right-skewed):

In [ ]:
use plotters_statistical::QqPlot;

evcxr_figure((760, 540), |root| {
    root.fill(&WHITE)?;
    for (area, name) in root.split_evenly((2, 2)).iter().zip(numeric.iter()) {
        let v = col_f64(&df, name)?;
        let (lo, hi) = (v.iter().cloned().fold(f64::INFINITY, f64::min),
                        v.iter().cloned().fold(f64::NEG_INFINITY, f64::max));
        let pad = (hi - lo) * 0.08 + 1e-9;
        let mut chart = ChartBuilder::on(area)
            .caption(*name, ("sans-serif", 15))
            .margin(6).x_label_area_size(28).y_label_area_size(46)
            .build_cartesian_2d(-3f64..3f64, (lo - pad)..(hi + pad))?;
        chart.configure_mesh().x_desc("theoretical").y_desc("sample").draw()?;
        chart.draw_series(std::iter::once(QqPlot::from_data(&v)?))?;
    }
    Ok(())
})

## 10. Line / time chart

`customers.csv` has **no real date column**, so a line-over-time chart on *it*
would manufacture structure that isn't there. But the chart type is worth showing,
so we illustrate it on a small **synthetic** monthly-signups series (genuine
time-indexed forecasting lives in the
[Time Series chapter](../10-time-series/time-series-fundamentals.ipynb)). A line
chart connects points along an **ordered** axis to reveal trend and seasonality:

In [ ]:
// Synthetic monthly new-signups: upward trend + yearly seasonality.
let signups: Vec<f64> = (0..24)
    .map(|m| 40.0 + m as f64 * 2.5 + 12.0 * ((m as f64) * std::f64::consts::PI / 6.0).sin())
    .collect();

evcxr_figure((720, 360), |root| {
    root.fill(&WHITE)?;
    let ymax = signups.iter().cloned().fold(0.0, f64::max) * 1.1;
    let mut chart = ChartBuilder::on(&root)
        .caption("New signups per month (synthetic)", ("sans-serif", 16))
        .margin(10).x_label_area_size(30).y_label_area_size(40)
        .build_cartesian_2d(0f64..23f64, 0f64..ymax)?;
    chart.configure_mesh().x_desc("month").y_desc("signups").draw()?;
    chart.draw_series(LineSeries::new((0..24).map(|m| (m as f64, signups[m])), BLUE.stroke_width(2)))?;
    chart.draw_series((0..24).map(|m| Circle::new((m as f64, signups[m]), 3, BLUE.filled())))?;
    Ok(())
})

(which-chart-for-which-question)=
## Which chart for which question

| Your question | Reach for |
| --- | --- |
| What's the shape/skew of one numeric variable? | **Histogram** or **ECDF** |
| Is one numeric variable normally distributed? | **Q–Q plot** |
| Where are the quartiles and outliers of one variable? | **Box plot** |
| What's the full density/shape (skew, modes) of one variable? | **Violin plot** |
| How strongly are many numeric variables related, at a glance? | **Correlation heatmap** |
| What does each pairwise relationship actually look like? | **Pair plot (scatter matrix)** |
| How frequent is each category? | **Bar chart (value counts)** |
| How balanced are the target classes? | **Bar chart (class balance)** |
| Which specific points are outliers, in context? | **Scatter with outliers flagged** |
| Do missing values cluster in particular rows/columns? | **Missingness heatmap** |
| How does a quantity evolve over an ordered axis? | **Line / time chart** (see Time Series) |

The box, violin, heatmap, pair-plot, ECDF and Q–Q charts here come from
[`plotters-statistical`](https://crates.io/crates/plotters-statistical); the rest
are `plotters`' own built-ins. The
[Model Evaluation](../01d-evaluation/cross-validation.ipynb) chapter reuses
several — the heatmap for confusion matrices, the bar chart for per-fold
scores — so they're worth getting comfortable with here first.